In [1]:
# imports

import requests
import zipfile
import io
import pandas as pd
import os

from datetime import datetime

StatementMeta(, 19a391d9-0abe-4f4e-a320-ac7110aa2f29, 3, Finished, Available, Finished, False)

In [2]:
# paths

save_dir = (
    "/lakehouse/default/Files/data/raw/"
    "nse_mcap"
)

os.makedirs(
    save_dir,
    exist_ok=True
)

StatementMeta(, 19a391d9-0abe-4f4e-a320-ac7110aa2f29, 4, Finished, Available, Finished, False)

In [3]:
# download report

url = (
    "https://www.nseindia.com/api/"
    "zipem?fileURLs=%5B"
    "%22https%3A%2F%2Fnsearchives.nseindia.com"
    "%2Farchives%2Fequities%2Fbhavcopy"
    "%2Fpr%2FPR050626.zip%22,"
    "%22https%3A%2F%2Fnsearchives.nseindia.com"
    "%2Fcontent%2Fcm%2F"
    "BhavCopy_NSE_CM_0_0_0_20260605_F_0000.csv.zip%22"
    "%5D&type=Daily"
)

headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.nseindia.com/"
}

response = requests.get(
    url,
    headers=headers,
    timeout=60
)

print(response.status_code)

StatementMeta(, 19a391d9-0abe-4f4e-a320-ac7110aa2f29, 5, Finished, Available, Finished, False)

200


In [4]:
# open outer zip

outer_zip = zipfile.ZipFile(
    io.BytesIO(response.content)
)

print(
    outer_zip.namelist()
)

StatementMeta(, 19a391d9-0abe-4f4e-a320-ac7110aa2f29, 6, Finished, Available, Finished, False)

['PR050626.zip', 'BhavCopy_NSE_CM_0_0_0_20260605_F_0000.csv.zip']


In [5]:
# find PR zip

pr_file = None

for file in outer_zip.namelist():

    if file.startswith("PR"):

        pr_file = file
        break

print(pr_file)

StatementMeta(, 19a391d9-0abe-4f4e-a320-ac7110aa2f29, 7, Finished, Available, Finished, False)

PR050626.zip


In [6]:
# open PR zip

pr_bytes = outer_zip.read(
    pr_file
)

pr_zip = zipfile.ZipFile(
    io.BytesIO(pr_bytes)
)

print(
    pr_zip.namelist()
)

StatementMeta(, 19a391d9-0abe-4f4e-a320-ac7110aa2f29, 8, Finished, Available, Finished, False)

['an05062026.txt', 'bc05062026.csv', 'bh05062026.csv', 'bm05062026.txt', 'corpbond05062026.csv', 'etf05062026.csv', 'gl05062026.csv', 'hl05062026.csv', 'mcap05062026.csv', 'pd05062026.csv', 'pr05062026.csv', 'readme.txt', 'sme05062026.csv', 'tt05062026.csv']


In [7]:
# extract mcap file

mcap_file = None

for file in pr_zip.namelist():

    if "mcap" in file.lower():

        mcap_file = file
        break

print(mcap_file)

StatementMeta(, 19a391d9-0abe-4f4e-a320-ac7110aa2f29, 9, Finished, Available, Finished, False)

mcap05062026.csv


In [8]:
# save csv

mcap_df = pd.read_csv(
    pr_zip.open(mcap_file)
)

save_path = os.path.join(
    save_dir,
    mcap_file
)

mcap_df.to_csv(
    save_path,
    index=False
)

print("SUCCESS")
print("Rows:", len(mcap_df))
print("Saved:", save_path)

StatementMeta(, 19a391d9-0abe-4f4e-a320-ac7110aa2f29, 10, Finished, Available, Finished, False)

SUCCESS
Rows: 2954
Saved: /lakehouse/default/Files/data/raw/nse_mcap/mcap05062026.csv


In [9]:
print(mcap_df.columns.tolist())
print(mcap_df.shape)

StatementMeta(, 19a391d9-0abe-4f4e-a320-ac7110aa2f29, 11, Finished, Available, Finished, False)

['Trade Date', 'Symbol', 'Series', 'Security Name', 'Category', 'Last Trade Date', 'Face Value(Rs.)', 'Issue Size', 'Close Price/Paid up value(Rs.)', 'Market Cap(Rs.)              ']
(2954, 10)


In [10]:
print(mcap_df.head())
print(mcap_df.shape)
print(mcap_df.columns.tolist())

StatementMeta(, 19a391d9-0abe-4f4e-a320-ac7110aa2f29, 13, Finished, Available, Finished, False)

    Trade Date      Symbol Series              Security Name    Category  \
0  05 JUN 2026   20MICRONS     EQ  20 MICRONS LTD             Listed       
1  05 JUN 2026  21STCENMGM     EQ  21ST CENTURY MGMT SERVICE  Listed       
2  05 JUN 2026      360ONE     EQ  360 ONE WAM LIMITED        Listed       
3  05 JUN 2026  3BBLACKBIO     EQ  3B BLACKBIO DX LTD.        Permitted    
4  05 JUN 2026   3IINFOLTD     EQ  3I INFOTECH LIMITED        Listed       

  Last Trade Date  Face Value(Rs.)  Issue Size  \
0     05 JUN 2026              5.0    35286502   
1     05 JUN 2026             10.0    10500000   
2     05 JUN 2026              1.0   405915969   
3     05 JUN 2026             10.0     8582670   
4     05 JUN 2026             10.0   207396267   

   Close Price/Paid up value(Rs.)  Market Cap(Rs.)                
0                          195.75                   6.908038e+09  
1                           33.00                   3.466050e+08  
2                         1073.90        

**BSE MCAP**

**BSE MCAP COLLECTION CLEANING AND SAVING **

In [ ]:
import pandas as pd
import requests

url = (
    "https://api.bseindia.com/BseIndiaAPI/api/"
    "ListofScripData/w"
    "?Group="
    "&Scripcode="
    "&segment=Equity"
    "&status="
)

headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.bseindia.com/"
}

data = requests.get(
    url,
    headers=headers
).json()

df = pd.DataFrame(data)

bse_mcap_df = df[
    [
        "SCRIP_CD",
        "scrip_id",
        "Issuer_Name",
        "ISIN_NUMBER",
        "Mktcap"
    ]
].copy()

bse_mcap_df["Mktcap"] = pd.to_numeric(
    bse_mcap_df["Mktcap"],
    errors="coerce"
)

bse_mcap_df = bse_mcap_df[
    bse_mcap_df["Mktcap"] > 0
]

bse_mcap_df.to_csv(
    "/lakehouse/default/Files/data/raw/bse_mcap/bse_mcap.csv",
    index=False
)

print("SUCCESS")
print("Rows:", len(bse_mcap_df))

In [2]:
%pip install yfinance

StatementMeta(, da14e7f5-5162-49c8-ade6-6211ea93882e, 9, Finished, Available, Finished, True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.8/137.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 51.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.2/146.2 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.6/184.6 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.6/215.6 kB 56.5 MB/s eta 0:00:00
  Attempting uninstall: websockets
    Found existing installation: websockets 12.0
    Not uninstalling websockets at /home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages, outside environment /nfs4/pyenv-26eed1ec-d92e-4aa9-a5f7-78352ab10fb8
    Can't uninstall 'websockets'. No files were found to uninstall.
  Attempting uninstall: cffi
    Found existing installation: cffi 1.16.0
    Not uninstalling cffi at /home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages, outside environment /nfs4/pyenv-26eed1ec-d92e-4aa9-a5f7-78352ab10fb8

In [3]:
import yfinance as yf

symbols = [
    "RELIANCE.NS",
    "TCS.NS",
    "INFY.NS",
    "HDFCBANK.NS"
]

for symbol in symbols:

    try:

        ticker = yf.Ticker(symbol)

        info = ticker.info

        print(symbol)
        print("Sector :", info.get("sector"))
        print("Industry :", info.get("industry"))
        print()

    except Exception as e:

        print(symbol, e)

StatementMeta(, da14e7f5-5162-49c8-ade6-6211ea93882e, 11, Finished, Available, Finished, False)

RELIANCE.NS
Sector : Energy
Industry : Oil & Gas Refining & Marketing

TCS.NS
Sector : Technology
Industry : Information Technology Services

INFY.NS
Sector : Technology
Industry : Information Technology Services

HDFCBANK.NS
Sector : Financial Services
Industry : Banks - Regional

